# Lab 01: Model Artifact Validation

**Audience:** Trainer

> **TRAINER ONLY.** Read this notebook, but do not start or change any server.

## Purpose
Validate the Qwen model directory on the CAI NFS workspace without downloading anything by default.

## Theory
The model weights live on shared NFS and are never packaged or downloaded by a notebook. This lab only checks that the directory is complete (config and safetensors files) and can record checksums. A download exists only as an explicit, trainer-only command-line flag.


## How to run
Run the cells from top to bottom; they are safe to run again.

- `LAB_MODE=local` uses explicit mocks or local services. Steps that need a GPU, CAI, Impala or NFS are reported as `SKIPPED` with a reason.
- `LAB_MODE=cai` uses the shared services from your environment. Nothing is mocked and a missing prerequisite is a `FAIL`.
- Save results only in your group workspace. Never paste secrets into a notebook.
- Participants do not run any deployment step in this lab.


In [ ]:
import json, os, sys
from pathlib import Path


def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'bni_training').is_dir():
            return candidate
    if os.getenv('BNI_REPO_ROOT'):
        return Path(os.environ['BNI_REPO_ROOT'])
    raise RuntimeError('Open this notebook inside the repository checkout, or set BNI_REPO_ROOT.')


ROOT = find_repo_root(Path.cwd().resolve())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
LAB_MODE = os.getenv('LAB_MODE', 'local').lower()
assert LAB_MODE in {'local', 'cai'}, 'LAB_MODE must be local or cai'
# A group workspace keeps its settings next to the notebooks: <workspace>/config/group.env
group_env = Path.cwd().parent / 'config' / 'group.env'
if group_env.is_file() and not os.getenv('GROUP_ENV_FILE'):
    os.environ['GROUP_ENV_FILE'] = str(group_env)
print({'lab_mode': LAB_MODE, 'python': sys.version.split()[0], 'group_env': 'found' if os.getenv('GROUP_ENV_FILE') else 'not found (local defaults)'})

In [ ]:
ROLE = os.getenv('WORKSHOP_ROLE', 'participant').lower()
print({'role': ROLE, 'trainer_only_lab': True})
if LAB_MODE == 'cai' and ROLE != 'trainer':
    print('You can read this notebook, but the steps are for the trainer. Do not start or change any server.')

In [ ]:
from bni_training.lab_registry import get_lab, load_script_module

lab = get_lab('01')
module = load_script_module(lab.script_path)
print(lab.id, lab.title, '-', 'TRAINER ONLY' if lab.trainer_only else 'participant lab')

## Run the lab
The script below is the canonical implementation. It prints one check per line and finishes with a `LAB_STATUS` line.


In [ ]:
exit_code = module.main(['--mode', LAB_MODE])
outcome = {0: 'PASS', 3: 'SKIPPED'}.get(exit_code, 'FAIL')
print({'exit_code': exit_code, 'outcome': outcome})
assert outcome != 'FAIL', 'The lab reported FAIL; read the checks above'

## Expected outcome
In CAI the model directory validates; locally a synthetic stand-in directory is checked and the NFS step is SKIPPED.

Every check is `PASS` or `SKIPPED` (with a reason). A `FAIL` stops the notebook.


## Checkpoint
Explain which component ran, which settings it used, and what you would expect if its service were unavailable.


## Optional challenge
Change one safe setting, rerun, and compare the results without touching shared services.


## Cleanup
Leave the shared Applications as the workshop phase requires; follow the Application lifecycle in the trainer docs.